# BAN 8101 Predictive Analytics
## Week 7: Model Selection, Interpretability & Error Analysis

### Big Question
**Once we have several models, how do we decide which one to trust, how do we explain it, and how do we learn from its mistakes?**

This week continues directly from Week 6 and uses the same self-driving steering problem:

$$
\text{Camera Image}\rightarrow\{\text{Left},\text{Straight},\text{Right}\}
$$

### Learning Objectives
After this lecture, you should be able to:
- Compare multiple models using the same cross-validation folds
- Explain why the highest score does not automatically identify the best model
- Compare predictive performance, stability, interpretability, and complexity
- Use built-in feature importance and permutation importance
- Inspect prediction confidence and uncertain cases
- Analyze confusion-matrix errors by steering class
- Investigate cases near steering-class boundaries
- Identify missed turns, unnecessary turns, and opposite-direction errors
- Build a model-selection scorecard
- Connect model selection and error analysis to the final self-driving simulation project

# 1. Why Week 7 Matters

In Week 6, we asked:

> **Which model predicts steering behavior best?**

This week, we ask:

> **Which model should we actually choose, and why?**

A practical model-selection decision may consider:

- predictive performance
- stability
- interpretability
- complexity
- runtime
- prediction confidence
- severity of errors
- deployment risk

For autonomous driving, a rare but severe steering error may matter more than a small improvement in average accuracy.

# 2. Dataset Setup

Use the same Udacity behavioral-cloning dataset from Week 6.

```text
data/
├── driving_log.csv
└── IMG/
```

Edit the local path below.

In [ ]:
from pathlib import Path

# -------------------------------------------------------------------
# STEP 1: Tell Python where the downloaded driving dataset is located.
#
# DATA_DIR should be the folder that contains:
#   - driving_log.csv
#   - the IMG folder
#
# Replace the example path below with the path on your own computer.
# Using Path makes file handling work more cleanly across Windows,
# macOS, and Linux.
# -------------------------------------------------------------------
DATA_DIR = Path(r"C:\CHANGE\THIS\PATH\data")

# Build the two locations we will use repeatedly later.
CSV_PATH = DATA_DIR / "driving_log.csv"
IMG_DIR = DATA_DIR / "IMG"

# Print the paths so we can visually confirm that they look correct
# before trying to load any files.
print("CSV:", CSV_PATH)
print("IMG:", IMG_DIR)

# 3. Import Libraries

In [ ]:
# -------------------------------------------------------------------
# Core numerical/data libraries
# -------------------------------------------------------------------
import numpy as np              # arrays and numerical calculations
import pandas as pd             # tabular data handling
import matplotlib.pyplot as plt # charts and image display
import cv2                      # OpenCV: image loading and preprocessing

# -------------------------------------------------------------------
# Model-validation tools
# -------------------------------------------------------------------
from sklearn.model_selection import (
    train_test_split,  # creates training and test sets
    StratifiedKFold,   # creates balanced cross-validation folds
    cross_validate     # evaluates one model across several folds
)

# -------------------------------------------------------------------
# Candidate predictive models from Week 6
# -------------------------------------------------------------------
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

# -------------------------------------------------------------------
# Model-interpretation tool
# -------------------------------------------------------------------
from sklearn.inspection import permutation_importance

# -------------------------------------------------------------------
# Evaluation tools
# -------------------------------------------------------------------
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
    classification_report
)

# Use one fixed random state so that train/test splitting and models
# can be reproduced when the notebook is rerun.
RANDOM_STATE = 42

# We intentionally limit the number of images used in this lecture
# so the notebook remains practical on a normal laptop.
# Set MAX_SAMPLES = None if you want to use all available images.
MAX_SAMPLES = 3000

print("Libraries loaded successfully.")

# 4. Reload the Driving Data

We reproduce the same Week 6 workflow so that Week 7 focuses on **model selection and interpretation**, not on changing the data pipeline.

In [ ]:
# -------------------------------------------------------------------
# STEP 1: Define the expected column names in driving_log.csv.
#
# The Udacity driving log normally stores three camera paths plus
# continuous driving-control values.
# -------------------------------------------------------------------
columns = [
    "center",
    "left",
    "right",
    "steering",
    "throttle",
    "brake",
    "speed"
]

# Load the CSV.
# header=None is used because many copies of this dataset do not
# contain a proper header row. We supply the names ourselves.
driving = pd.read_csv(
    CSV_PATH,
    names=columns,
    header=None
)

# -------------------------------------------------------------------
# STEP 2: Make sure the steering column is numeric.
#
# If a text header such as "steering" is accidentally present in the
# first row, errors="coerce" converts it to NaN rather than crashing.
# We then remove rows whose steering value could not be converted.
# -------------------------------------------------------------------
driving["steering"] = pd.to_numeric(
    driving["steering"],
    errors="coerce"
)

driving = driving.dropna(
    subset=["steering"]
).copy()

# -------------------------------------------------------------------
# STEP 3: Repair image paths.
#
# The CSV may contain an old absolute path from the computer on which
# the data were originally recorded.
#
# Example:
#   C:\old_user\data\IMG\center_....jpg
#
# We only need the filename. We reconnect that filename to our own
# local IMG folder.
# -------------------------------------------------------------------
def local_image_path(recorded_path):
    # Convert Windows "\" separators to "/" so the same logic works
    # even if the path came from another operating system.
    cleaned = str(recorded_path).strip().replace("\\", "/")

    # Keep only the final filename.
    filename = cleaned.split("/")[-1]

    # Build a valid local path on this computer.
    return IMG_DIR / filename

driving["center_path"] = driving["center"].apply(
    local_image_path
)

# -------------------------------------------------------------------
# STEP 4: Verify that the image files actually exist.
#
# Rows whose center image cannot be found are removed because we cannot
# build image features from a missing file.
# -------------------------------------------------------------------
exists = driving["center_path"].apply(
    lambda p: Path(p).exists()
)

driving = driving.loc[exists].copy()

print("Rows available:", len(driving))

# 5. Recreate the Steering Classes

We use the same Week 6 target definition:

$$
\text{angle}<-0.05\Rightarrow \text{Left}
$$

$$
-0.05\leq\text{angle}\leq0.05\Rightarrow\text{Straight}
$$

$$
\text{angle}>0.05\Rightarrow\text{Right}
$$

In [ ]:
# -------------------------------------------------------------------
# The original steering variable is continuous.
# For this Week 7 classification exercise, we convert it into:
#   Left / Straight / Right
#
# A threshold of 0.05 means small steering values near zero are
# considered "Straight".
# -------------------------------------------------------------------
STEERING_THRESHOLD = 0.05

def steering_class(angle, threshold=STEERING_THRESHOLD):
    """
    Convert one continuous steering angle into a categorical label.

    Negative angles below -threshold  -> Left
    Positive angles above +threshold  -> Right
    Values between the two thresholds -> Straight
    """
    if angle < -threshold:
        return "Left"
    elif angle > threshold:
        return "Right"
    return "Straight"

# Apply the function to every steering value in the dataset.
driving["steering_class"] = driving["steering"].apply(
    steering_class
)

# -------------------------------------------------------------------
# Inspect class balance.
#
# Count = number of observations in each class
# Proportion = fraction of the full dataset in each class
#
# If Straight is much more common than Left/Right, overall accuracy can
# become misleading. That is why we also use Macro F1 later.
# -------------------------------------------------------------------
display(
    pd.DataFrame({
        "Count": driving["steering_class"].value_counts(),
        "Proportion":
            driving["steering_class"]
            .value_counts(normalize=True)
            .round(3)
    })
)

## Why Macro F1?

If Straight is more common than Left or Right, accuracy alone may hide weak performance on the turning classes.

$$
F1_{\text{macro}}
=
\frac{
F1_{\text{Left}}+
F1_{\text{Straight}}+
F1_{\text{Right}}
}{3}
$$

# 6. Recreate the Week 6 Image Preprocessing

In [ ]:
# -------------------------------------------------------------------
# Image preprocessing settings
#
# We reduce each camera image to 32 x 16 grayscale pixels.
# That creates only 512 predictors instead of tens of thousands.
# -------------------------------------------------------------------
RESIZE_WIDTH = 32
RESIZE_HEIGHT = 16

def read_rgb(path):
    """
    Load an image for DISPLAY.

    OpenCV reads images in BGR order by default.
    Matplotlib expects RGB, so we convert the color ordering.
    """
    image = cv2.imread(str(path))

    if image is None:
        raise FileNotFoundError(path)

    return cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

def preprocess_image(path):
    """
    Convert one road image into the numerical representation used
    by our tree models.

    Steps:
    1. Load image
    2. Crop much of the sky/dashboard
    3. Convert to grayscale
    4. Resize to 32 x 16
    5. Scale pixel values to the 0-1 range
    """
    img = cv2.imread(str(path))

    if img is None:
        raise FileNotFoundError(path)

    # Original image dimensions
    h, w = img.shape[:2]

    # Focus on the road region rather than the full image.
    cropped = img[
        int(h * 0.35):int(h * 0.90),
        :
    ]

    # Grayscale reduces three color channels to one channel.
    gray = cv2.cvtColor(
        cropped,
        cv2.COLOR_BGR2GRAY
    )

    # Resize every image to exactly the same dimensions.
    resized = cv2.resize(
        gray,
        (RESIZE_WIDTH, RESIZE_HEIGHT),
        interpolation=cv2.INTER_AREA
    )

    # Convert integer pixel values (0-255) to floating values (0-1).
    return resized.astype(np.float32) / 255.0

In [ ]:
# -------------------------------------------------------------------
# Keep only the columns needed for this Week 7 analysis.
# -------------------------------------------------------------------
model_df = driving[
    ["center_path", "steering", "steering_class"]
].copy()

# -------------------------------------------------------------------
# Optional sampling for speed.
#
# If more than MAX_SAMPLES images exist, keep a stratified sample so
# that Left/Straight/Right remain represented in roughly the same
# proportions as the full dataset.
# -------------------------------------------------------------------
if MAX_SAMPLES is not None and len(model_df) > MAX_SAMPLES:
    model_df, _ = train_test_split(
        model_df,
        train_size=MAX_SAMPLES,
        random_state=RANDOM_STATE,
        stratify=model_df["steering_class"]
    )

# These lists will collect the information for each observation.
features = []  # flattened 512-pixel feature vectors
labels = []    # Left / Straight / Right
angles = []    # original continuous steering values
paths = []     # original image paths for later visual inspection

# -------------------------------------------------------------------
# Process every selected image.
# -------------------------------------------------------------------
for _, row in model_df.iterrows():

    # Convert image into a 16 x 32 numerical matrix,
    # then flatten it into one vector of length 512.
    feature_vector = preprocess_image(
        row["center_path"]
    ).flatten()

    features.append(feature_vector)
    labels.append(row["steering_class"])
    angles.append(row["steering"])
    paths.append(row["center_path"])

# Convert Python lists into NumPy arrays, which scikit-learn expects.
X = np.vstack(features)
y = np.asarray(labels)
angles = np.asarray(angles, dtype=float)
paths = np.asarray(paths, dtype=object)

# Expected shape:
#   X -> (number_of_images, 512)
#   y -> (number_of_images,)
print("X:", X.shape)
print("y:", y.shape)

# 7. Create a Consistent Train/Test Split

In [ ]:
# -------------------------------------------------------------------
# Split the observations into training and test sets.
#
# We split X, y, the continuous steering angle, and the file path
# together so every piece of information stays aligned by observation.
# -------------------------------------------------------------------
(
    X_train, X_test,
    y_train, y_test,
    angle_train, angle_test,
    path_train, path_test
) = train_test_split(
    X,
    y,
    angles,
    paths,

    # Reserve 20% of the observations for final testing.
    test_size=0.20,

    # Reproducible split.
    random_state=RANDOM_STATE,

    # Preserve the Left/Straight/Right proportions.
    stratify=y
)

print("Training observations:", len(y_train))
print("Testing observations :", len(y_test))

# 8. Fair Model Comparison

All candidate models will use the **same five cross-validation folds**.

$$
\text{Same Data}+\text{Same CV Folds}\Rightarrow\text{Fairer Comparison}
$$

In [ ]:
# -------------------------------------------------------------------
# Create ONE cross-validation design and reuse it for every model.
#
# Why StratifiedKFold?
# Each fold should contain roughly similar proportions of:
#   Left / Straight / Right
#
# Why shuffle=True?
# It avoids relying on the original row order of the dataset.
#
# Why use the same CV object for every model?
# It makes the comparison fairer because each model is tested on the
# same validation observations.
# -------------------------------------------------------------------
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# 9. Candidate Models

In [ ]:
# -------------------------------------------------------------------
# Candidate Model 1: Decision Tree
#
# max_depth=8 limits how complex the tree can become.
# class_weight="balanced" increases the influence of less-common classes.
# -------------------------------------------------------------------
tree_model = DecisionTreeClassifier(
    max_depth=8,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# -------------------------------------------------------------------
# Candidate Model 2: Random Forest
#
# n_estimators=150 means we combine 150 decision trees.
# n_jobs=-1 asks scikit-learn to use all available CPU cores.
# -------------------------------------------------------------------
rf_model = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# -------------------------------------------------------------------
# Candidate Model 3: Gradient Boosting
#
# n_estimators = number of sequential trees
# learning_rate = strength of each new correction
# max_depth = complexity of each small tree
# -------------------------------------------------------------------
gb_model = GradientBoostingClassifier(
    n_estimators=60,
    learning_rate=0.08,
    max_depth=2,
    random_state=RANDOM_STATE
)

# Store all candidate models in one dictionary so we can evaluate them
# using exactly the same code and cross-validation design.
models = {
    "Decision Tree": tree_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model
}

# 10. Cross-Validation Comparison

We compare:
- Mean CV Accuracy
- Mean CV Macro F1
- CV Macro F1 standard deviation

The mean estimates typical performance.  
The standard deviation describes stability across folds.

In [ ]:
# -------------------------------------------------------------------
# Compare every candidate model using the SAME 5 CV folds.
#
# We collect:
#   1. Accuracy
#   2. Macro F1
#   3. Variability of Macro F1 across folds
#
# Macro F1 is especially important because steering classes may be
# imbalanced.
# -------------------------------------------------------------------
cv_rows = []

for name, model in models.items():

    result = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,

        # Request more than one evaluation metric at the same time.
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro"
        },

        # Tree ensembles can use multiple CPU cores.
        # GradientBoostingClassifier itself is not parallelized here.
        n_jobs=-1 if name != "Gradient Boosting" else None
    )

    # Store one summary row per model.
    cv_rows.append({
        "Model": name,

        # Average predictive performance across the five validation folds.
        "Mean CV Accuracy":
            result["test_accuracy"].mean(),

        "Mean CV Macro F1":
            result["test_f1_macro"].mean(),

        # Standard deviation tells us how stable/variable performance was.
        "CV Macro F1 Std":
            result["test_f1_macro"].std()
    })

# Create an easy-to-read comparison table.
# Sort by Macro F1 because it treats all three steering classes equally.
cv_results = pd.DataFrame(
    cv_rows
).sort_values(
    "Mean CV Macro F1",
    ascending=False
)

cv_results.round(3)

## Pause and Think

If two models differ by only 0.01 in Macro F1, but one is far less stable or much harder to explain, should the slightly higher-scoring model automatically win?

**No.**

Model selection is a multi-criteria decision.

# 11. What Does "Best" Mean?

| Criterion | Question |
|---|---|
| Performance | Does it predict well? |
| Stability | Does performance remain similar across folds? |
| Interpretability | Can we explain what it uses? |
| Complexity | How hard is it to tune and maintain? |
| Failure behavior | What kinds of mistakes occur? |
| Confidence | Does it reveal uncertain predictions? |
| Deployment risk | Are severe errors acceptable? |

# 12. Fit All Candidate Models

In [ ]:
# -------------------------------------------------------------------
# cross_validate() trains temporary model copies internally.
#
# For the interpretation/error-analysis sections below, we need our own
# three model objects fitted on the FULL training set.
# -------------------------------------------------------------------
for model in models.values():
    model.fit(
        X_train,
        y_train
    )

print("All candidate models fitted.")

# 13. Global Interpretability: Built-In Random Forest Importance

Each processed image has:

$$
32\times16=512
$$

pixel features.

Random Forest's built-in feature importance measures how much each feature contributed to impurity reduction during training.

In [ ]:
# -------------------------------------------------------------------
# Random Forest's built-in feature importance returns one value for
# each of our 512 pixel features.
# -------------------------------------------------------------------
pixel_importance = rf_model.feature_importances_

# Reshape the 512 importance values back into a 16 x 32 image.
# This lets us see WHERE in the road image the model relied more heavily.
rf_importance = pixel_importance.reshape(
    RESIZE_HEIGHT,
    RESIZE_WIDTH
)

# Display the importance values as a heatmap.
plt.figure(figsize=(10, 5))
plt.imshow(
    rf_importance,
    cmap="hot"
)
plt.colorbar(
    label="Impurity-Based Importance"
)
plt.title(
    "Random Forest Built-In Pixel Importance"
)
plt.axis("off")
plt.show()

### Interpretation Warning

A high importance value means:

> **The model relied heavily on this feature.**

It does not mean:

> **The feature causes the steering behavior.**

# 14. Permutation Importance

Permutation importance asks:

> **How much does model performance decrease when the information in one feature is destroyed by shuffling it?**

$$
Importance_j
=
Score_{\text{baseline}}
-
Score_{\text{shuffled }j}
$$

A larger drop suggests stronger dependence on that feature.

In [ ]:
# -------------------------------------------------------------------
# Permutation importance can be computationally expensive because each
# feature is shuffled repeatedly and the model is re-scored.
#
# We use at most 500 test observations to keep lecture runtime practical.
# -------------------------------------------------------------------
perm_n = min(
    500,
    len(X_test)
)

# -------------------------------------------------------------------
# Calculate permutation importance for the fitted Random Forest.
#
# For each pixel:
#   1. measure baseline Macro F1
#   2. shuffle that pixel column
#   3. measure Macro F1 again
#   4. calculate the performance drop
#
# n_repeats=3 repeats the shuffle three times to reduce dependence on
# one random permutation.
# -------------------------------------------------------------------
perm = permutation_importance(
    rf_model,
    X_test[:perm_n],
    y_test[:perm_n],
    scoring="f1_macro",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Each pixel has an average importance across the repeated shuffles.
# Reshape the 512 values back into the processed image layout.
perm_map = perm.importances_mean.reshape(
    RESIZE_HEIGHT,
    RESIZE_WIDTH
)

plt.figure(figsize=(10, 5))
plt.imshow(
    perm_map,
    cmap="hot"
)
plt.colorbar(
    label="Permutation Importance"
)
plt.title(
    "Random Forest Permutation Pixel Importance"
)
plt.axis("off")
plt.show()

## Check Your Understanding

Compare the two importance maps.

- Do they emphasize similar road regions?
- Why might neighboring correlated pixels reduce permutation importance?
- Why should neither importance method be interpreted as causation?

# 15. Global vs. Local Interpretability

### Global
What does the model generally rely on?

### Local
Why did the model make this prediction for this specific road image?

Week 7 focuses on both levels because future autonomous-driving models will become harder to interpret.

# 16. Prediction Confidence

A classifier may predict:

```text
Left       0.05
Straight   0.90
Right      0.05
```

or:

```text
Left       0.32
Straight   0.36
Right      0.32
```

Both return one class, but the second prediction is much more uncertain.

In [ ]:
# -------------------------------------------------------------------
# predict_proba() gives one probability for each possible steering class.
#
# For three classes, rf_prob has shape:
#   (number_of_test_images, 3)
#
# Example row:
#   [0.10, 0.75, 0.15]
# -------------------------------------------------------------------
rf_prob = rf_model.predict_proba(
    X_test
)

# predict() returns only the final class label.
rf_pred = rf_model.predict(
    X_test
)

# -------------------------------------------------------------------
# Define "confidence" as the model's largest class probability.
#
# Example:
#   [0.10, 0.75, 0.15] -> confidence = 0.75
#
# Important:
# High confidence does NOT guarantee the prediction is correct.
# -------------------------------------------------------------------
confidence = rf_prob.max(
    axis=1
)

# Build a table connecting each actual class, prediction, and confidence.
confidence_df = pd.DataFrame({
    "Actual": y_test,
    "Prediction": rf_pred,
    "Confidence": confidence
})

confidence_df.head()

# 17. Inspect the Most Uncertain Predictions

In [ ]:
# -------------------------------------------------------------------
# argsort() returns indices ordered from smallest to largest confidence.
#
# Taking the first 12 therefore gives the model's 12 most uncertain
# test predictions.
# -------------------------------------------------------------------
uncertain_idx = np.argsort(
    confidence
)[:12]

# Create a 3 x 4 display grid.
fig, axes = plt.subplots(
    3,
    4,
    figsize=(12, 8)
)

for ax, idx in zip(
    axes.ravel(),
    uncertain_idx
):
    # Recover the ORIGINAL road image using its saved file path.
    ax.imshow(
        read_rgb(path_test[idx])
    )

    # Show:
    #   actual steering class
    #   predicted steering class
    #   maximum predicted probability
    ax.set_title(
        f"Actual: {y_test[idx]}\n"
        f"Pred: {rf_pred[idx]}\n"
        f"Conf: {confidence[idx]:.2f}"
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

## Pause and Think

Would you want a self-driving system to treat a 36% confident steering prediction the same way as a 95% confident prediction?

Confidence does not guarantee correctness, but it helps identify uncertain situations.

# 18. Compare Confusion Matrices

In [ ]:
# -------------------------------------------------------------------
# Display one confusion matrix for each candidate model side-by-side.
#
# Using the same test set makes the error patterns directly comparable.
# -------------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

for ax, (name, model) in zip(
    axes,
    models.items()
):
    # Generate each model's predictions on the common test set.
    pred = model.predict(
        X_test
    )

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        pred,

        # Keep row/column class order consistent across all matrices.
        labels=[
            "Left",
            "Straight",
            "Right"
        ],

        ax=ax,
        cmap="Blues",
        colorbar=False
    )

    ax.set_title(name)

plt.tight_layout()
plt.show()

# 19. Self-Driving Error Categories

### Missed Turn
Actual turn, predicted Straight.

### Unnecessary Turn
Actual Straight, predicted Left or Right.

### Opposite-Direction Error
Actual Left, predicted Right — or vice versa.

These errors may have different practical severity.

In [ ]:
def steering_error_category(actual, predicted):
    """
    Translate a generic classification mistake into a more meaningful
    self-driving failure category.
    """

    # Correct steering class
    if actual == predicted:
        return "Correct"

    # A real turn was needed, but the model chose Straight.
    if (
        actual in ["Left", "Right"]
        and predicted == "Straight"
    ):
        return "Missed Turn"

    # The road was labeled Straight, but the model introduced a turn.
    if (
        actual == "Straight"
        and predicted in ["Left", "Right"]
    ):
        return "Unnecessary Turn"

    # The model steered in the opposite direction from the label.
    if (
        (actual == "Left" and predicted == "Right")
        or
        (actual == "Right" and predicted == "Left")
    ):
        return "Opposite Direction"

    return "Other"

# -------------------------------------------------------------------
# Count each kind of failure for every candidate model.
# -------------------------------------------------------------------
rows = []

for name, model in models.items():

    pred = model.predict(
        X_test
    )

    # Convert each actual/predicted pair into a domain-specific category.
    categories = [
        steering_error_category(a, p)
        for a, p in zip(y_test, pred)
    ]

    counts = pd.Series(
        categories
    ).value_counts()

    rows.append({
        "Model": name,
        "Correct":
            counts.get("Correct", 0),
        "Missed Turn":
            counts.get("Missed Turn", 0),
        "Unnecessary Turn":
            counts.get("Unnecessary Turn", 0),
        "Opposite Direction":
            counts.get("Opposite Direction", 0)
    })

error_summary = pd.DataFrame(
    rows
)

error_summary

## What to Notice

A model can have slightly better Macro F1 but still make more severe opposite-direction errors.

That is why domain consequences belong in model selection.

# 20. Steering-Threshold Boundary Analysis

Our three classes were created from a continuous steering angle using:

$$
\pm0.05
$$

Consider:

```text
0.049 → Straight
0.051 → Right
```

Those values are almost identical physically, yet they receive different labels.

Some apparent classification errors may therefore be **boundary ambiguities**.

In [ ]:
# -------------------------------------------------------------------
# Our labels were created from two boundaries:
#   -0.05 and +0.05
#
# We now calculate how far every test observation is from the NEAREST
# boundary.
#
# A very small distance means the original continuous steering angle was
# close to switching from one class label to another.
# -------------------------------------------------------------------
boundary_distance = np.minimum(
    np.abs(
        angle_test - STEERING_THRESHOLD
    ),
    np.abs(
        angle_test + STEERING_THRESHOLD
    )
)

# Boolean vector:
# True  = Random Forest prediction is correct
# False = Random Forest prediction is wrong
rf_correct = (
    rf_pred == y_test
)

# Combine the original continuous steering value, classification result,
# and boundary distance into one analysis table.
boundary_df = pd.DataFrame({
    "Steering Angle": angle_test,
    "Actual Class": y_test,
    "Predicted Class": rf_pred,
    "Correct": rf_correct,
    "Distance to Boundary":
        boundary_distance
})

# Show the observations nearest to a class boundary first.
boundary_df.sort_values(
    "Distance to Boundary"
).head(15)

# 21. Are Errors More Common Near the Boundary?

In [ ]:
# -------------------------------------------------------------------
# Define "near the boundary" as within 0.03 steering units of either
# -0.05 or +0.05.
#
# This is an analytical choice for exploration, not a universal rule.
# -------------------------------------------------------------------
NEAR_BOUNDARY = 0.03

near_mask = (
    boundary_distance <= NEAR_BOUNDARY
)

far_mask = (
    boundary_distance > NEAR_BOUNDARY
)

# rf_correct is True/False.
# Its mean is therefore the accuracy within that subset.
# 1 - accuracy gives the corresponding error rate.
near_error_rate = (
    1 - rf_correct[near_mask].mean()
)

far_error_rate = (
    1 - rf_correct[far_mask].mean()
)

print(
    "Near-boundary error rate:",
    round(near_error_rate, 3)
)

print(
    "Far-from-boundary error rate:",
    round(far_error_rate, 3)
)

## Interpretation

If errors are more common near the thresholds, part of the difficulty may come from the **target definition**, not only from the algorithm.

$$
\text{Model Quality}\neq\text{Only Algorithm Quality}
$$

Label design, preprocessing, and evaluation choices matter too.

# 22. Visual Failure Analysis

In [ ]:
# -------------------------------------------------------------------
# Locate every Random Forest test prediction that was incorrect.
# -------------------------------------------------------------------
wrong_idx = np.where(
    rf_pred != y_test
)[0]

# Display at most 12 mistakes so the output remains readable.
n_show = min(
    12,
    len(wrong_idx)
)

fig, axes = plt.subplots(
    3,
    4,
    figsize=(12, 8)
)

for ax, idx in zip(
    axes.ravel(),
    wrong_idx[:n_show]
):
    # Translate the mistake into a driving-specific failure category.
    category = steering_error_category(
        y_test[idx],
        rf_pred[idx]
    )

    # Display the ORIGINAL image rather than the tiny 32x16 input so
    # humans can inspect the road scene.
    ax.imshow(
        read_rgb(path_test[idx])
    )

    ax.set_title(
        f"{category}\n"
        f"Actual: {y_test[idx]}\n"
        f"Pred: {rf_pred[idx]}\n"
        f"Angle: {angle_test[idx]:.3f}"
    )

    ax.axis("off")

# Hide unused subplot panels if fewer than 12 errors exist.
for ax in axes.ravel()[n_show:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Error Analysis Questions

- Is the road visibly curved?
- Is the steering angle near the class boundary?
- Are lane markings visible?
- Are shadows present?
- Could the driver's steering action reflect recovery behavior?
- Is one image sufficient to determine the ideal steering action?

Move from:

> "The model made an error."

to:

> **"Why might the model have made this error?"**

# 23. Model Selection Scorecard

In [ ]:
# -------------------------------------------------------------------
# Start the scorecard with the quantitative cross-validation results.
# -------------------------------------------------------------------
scorecard = cv_results.copy()

# -------------------------------------------------------------------
# Add qualitative model properties.
#
# These labels are discussion aids, not mathematically estimated values.
# Students should recognize that interpretability and complexity involve
# judgment and depend on the deployment context.
# -------------------------------------------------------------------
scorecard["Interpretability"] = scorecard["Model"].map({
    "Decision Tree": "High",
    "Random Forest": "Medium",
    "Gradient Boosting": "Medium-Low"
})

scorecard["Complexity"] = scorecard["Model"].map({
    "Decision Tree": "Low",
    "Random Forest": "High",
    "Gradient Boosting": "High"
})

# -------------------------------------------------------------------
# Merge domain-specific error counts into the same table.
#
# This lets us compare:
#   predictive performance
#   stability
#   interpretability
#   complexity
#   driving-error severity
#
# rather than selecting a model from only one metric.
# -------------------------------------------------------------------
scorecard = scorecard.merge(
    error_summary[[
        "Model",
        "Missed Turn",
        "Unnecessary Turn",
        "Opposite Direction"
    ]],
    on="Model",
    how="left"
)

scorecard.round(3)

# 24. Make a Recommendation

A defensible recommendation should consider:

1. validation performance
2. stability
3. interpretability
4. complexity
5. severe error counts
6. uncertainty
7. intended deployment context

### Recommendation Template

> I recommend __________ because __________.
>
> Its mean CV Macro F1 was __________ with a standard deviation of __________.
>
> Its strongest advantage is __________.
>
> Its most important limitation is __________.
>
> Before using it in an autonomous-driving system, I would investigate __________.

# 25. Why Interpretability Matters in Autonomous Driving

If a model predicts **Turn Right**, we may want to know:

- which visual regions mattered
- whether the model was confident
- whether the scene resembles known failure cases
- whether the decision was near a label boundary
- whether the model makes systematic errors

As models become more powerful, they often become harder to interpret.

That leads directly into neural networks.

# 26. Final Project Team-Up

Week 7 is also the beginning of final-project planning.

Your final project will integrate:

$$
\text{Perception}\rightarrow\text{Prediction / Decision}\rightarrow\text{Vehicle Control}
$$

Complete the planning prompts below with your team.

## Team Planning Worksheet

### A. Team Members
List all members.

### B. Communication Plan
How will you communicate and how often will you meet?

### C. Tentative Technical Approach
- Behavioral Cloning
- Reinforcement Learning
- Hybrid
- Undecided

### D. Preliminary Roles
Possible responsibilities:
- data / preprocessing
- modeling
- simulator integration
- validation / testing
- documentation
- presentation

### E. Definition of Success
Possible measures:
- route completion
- crashes
- lane departures
- steering smoothness
- robustness across runs

### F. Validation Plan
How will you test the vehicle under conditions not used during development?

### G. Risks
Identify at least three project risks.

# 27. Bridge to Week 8 — Neural Network Foundations

So far, tree models treat the processed road image as:

$$
x_1,x_2,\ldots,x_{512}
$$

Next week, we begin **Neural Network Foundations**.

Later, CNNs will move us toward:

$$
\text{Raw Camera Image}
\rightarrow
\text{Learned Visual Features}
\rightarrow
\text{Steering Prediction}
$$

That is a major step toward the final self-driving simulation.

# 28. Week 7 Takeaways

- Model selection should use more than one metric.
- Cross-validation mean and variability both matter.
- Interpretability and complexity matter alongside predictive performance.
- Built-in importance and permutation importance answer different questions.
- Confidence helps identify uncertain predictions.
- Confusion matrices reveal which steering behaviors are confused.
- Self-driving errors differ in severity.
- Some apparent errors may come from class-boundary definitions.
- Failure analysis helps us understand why models struggle.
- The best model is the one most appropriate for the actual decision context.

### Key Principle

> **A strong predictive model is not merely accurate. It should also be stable, understandable enough for its purpose, and evaluated according to the consequences of its failures.**

# Reflection Questions

1. Why is the highest-scoring model not automatically the best choice?
2. What does CV standard deviation tell you?
3. What is the difference between global and local interpretability?
4. How does permutation importance differ from built-in tree importance?
5. Why can correlated pixels complicate importance interpretation?
6. What does low prediction confidence tell you?
7. Which steering error type is most serious and why?
8. Why might near-boundary observations be more difficult?
9. Which model would you recommend and why?
10. Which Week 7 evaluation ideas will matter most in the final self-driving project?